In [1]:
!pip install -U --no-cache-dir gdown
import gdown
url='https://drive.google.com/file/d/1SA6OA8JILLuv-eOcPkUNbgPNopTgQ-Bx/view?usp=sharing'
gdown.download(url, 'final_dataset.jsonl', quiet=False)

  Attempting uninstall: gdown
    Found existing installation: gdown 5.2.1
    Uninstalling gdown-5.2.1:
      Successfully uninstalled gdown-5.2.1


Downloading...
From: https://drive.google.com/uc?id=1SA6OA8JILLuv-eOcPkUNbgPNopTgQ-Bx
To: /kaggle/working/final_dataset.jsonl
100%|██████████| 6.47M/6.47M [00:00<00:00, 77.0MB/s]


'final_dataset.jsonl'

In [5]:
# Gỡ cài đặt các bản cũ có thể gây xung đột
!pip uninstall -y bitsandbytes unsloth torch torchvision torchaudio

# Cài đặt phiên bản Unsloth tối ưu cho Kaggle (thường dùng CUDA 12.1 hoặc 11.8)
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-cache-dir --force-reinstall --no-deps bitsandbytes torch torchvision unsloth trl transformers peft

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Found existing installation: unsloth 2026.4.5
Uninstalling unsloth-2026.4.5:
  Successfully uninstalled unsloth-2026.4.5
Found existing installation: torch 2.11.0
Uninstalling torch-2.11.0:
  Successfully uninstalled torch-2.11.0
Found existing installation: torchvision 0.26.0
Uninstalling torchvision-0.26.0:
  Successfully uninstalled torchvision-0.26.0
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-xqhs76cp/unsloth_68c3bd569a254a86bd38bf3db572138a
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-xqhs76cp/unsloth_68c3bd569a254a86bd38bf3db572138a
  Resolved https://github.com/unslothai/unsloth.git to commit 14ab6fbfae79b9b8ee8612793ecd3f2fac528d93
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) 

In [9]:
from unsloth import FastLanguageModel

/tmp/ipykernel_55/2814113929.py:1: UserWarning: WARNING: Unsloth should be imported before [trl, transformers, peft] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from unsloth import FastLanguageModel


RuntimeError: operator torchvision::nms does not exist

In [7]:
import torch
import json
from datasets import Dataset

max_seq_length = 8192 # Tăng lên để chứa các file code dài
dtype = None # Tự động phát hiện (T4 dùng Float16)
load_in_4bit = True # Tiết kiệm VRAM

In [8]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Thêm LoRA Adapters (Unsloth tối ưu hóa các lớp này để train nhanh gấp 2x)
model = FastLanguageModel.get_peft_model(
    model,
    r = 8,
    lora_alpha = 16,
    # lora_dropout = 0.15,
    target_modules = [
        "q_proj","k_proj","v_proj","o_proj",
        "gate_proj","up_proj","down_proj"
    ],
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    use_rslora = True,
)

model.config.attention_dropout = 0.1
model.config.hidden_dropout = 0.1

NameError: name 'FastLanguageModel' is not defined

In [ ]:
ALT_INSTRUCTIONS = [
    # 1. Phong cách Chuyên gia Kiến trúc (Architecture Expert)
    """Act as a Senior Software Architect. Your goal is to perform a structural audit on the provided Java/C++ method.
Analyze the source code, static metrics, and evolution history to identify two specific architectural smells:
- SHOTGUN SURGERY: Occurs when a change in this method forces ripples across many external classes (Requires: fan_out >= [TH_FAN_OUT], co_changed_classes >= [TH_CO_CLASSES], commit_count >= [TH_COMMIT_MIN]).
- DIVERGENT CHANGE: Occurs when the method is a 'God Method' modified for too many unrelated reasons (Requires: commit_count >= [TH_COMMIT_COUNT], distinct_concerns >= 3, complexity_delta > 0).

Evaluate the 'external_calls' for coupling and commit logs for concern scattering. Provide derived metrics, a semantic deep-dive, and a refactoring plan.
Output format: Strictly a single JSON object following the established schema.""",

    # 2. Phong cách Máy phân tích dữ liệu (Data-Driven Analyzer)
    """You are an AI-powered static analysis tool. Process the following method data to detect design violations.
Criteria for SHOTGUN SURGERY: (fan_out >= [TH_FAN_OUT]) AND (total_unique_co_changed_classes >= [TH_CO_CLASSES]) AND (commit_count >= [TH_COMMIT_MIN]).
Criteria for DIVERGENT CHANGE: (commit_count >= [TH_COMMIT_COUNT]) AND (distinct_concerns >= 3) AND (complexity_delta > 0).

Instructions:
1. Parse the method body and external_calls.
2. Cross-reference metrics against thresholds.
3. Identify distinct feature areas in commit messages.
4. Resolve conflicts where metrics might be overstated due to global refactors.
Return ONLY a valid JSON object. No prose, no markdown.""",

    # 3. Phong cách Hướng dẫn quy trình (Process-Oriented)
    """Follow this protocol to evaluate code smells in the given Java/C++ snippet:
Step 1: Extract metrics from 'pre_computed' and 'static_analysis'.
Step 2: Calculate all derived_metrics (churn, frequency, complexity delta).
Step 3: Check SHOTGUN SURGERY: Does the method have high coupling and frequent co-changes?
Step 4: Check DIVERGENT CHANGE: Is the method growing in complexity due to unrelated feature requests?
Step 5: Synthesize a reasoning chain to justify the final label and confidence score.

Output Requirement: A raw JSON object containing derived_metrics, semantic_analysis, shotgun_surgery, divergent_change, final_decision, and suggested_refactor.""",

    # 4. Phong cách Thẩm định Code (Code Auditor)
    """Perform a technical audit on the method below. Focus on coupling and cohesion.
Target Smells:
- Shotgun Surgery (High fan-out + High co-change count).
- Divergent Change (High change frequency + Multiple concerns).

Compare the extracted data against thresholds [TH_FAN_OUT, TH_CO_CLASSES, TH_COMMIT_MIN, TH_COMMIT_COUNT]. Use 'external_calls' to verify coupling intensity.
If the metrics meet thresholds but the semantics show a stable configuration or a global version upgrade, downgrade the smell probability.
Return a JSON object only.""",

    # 5. Phong cách Kiểm thử Tự động (Automated Testing Agent)
    """Execute code smell detection on the input method using git history and static metrics.
Metrics to compute: avg_churn_per_commit, change_frequency_monthly, and complexity_delta.
Threshold validation:
- Label as 'shotgun_surgery' if metrics for fan_out and co-changes are satisfied.
- Label as 'divergent_change' if metrics for commit count and concern keywords are satisfied.

Provide a concise refactoring suggestion if a smell is detected, otherwise suggest 'No changes required'.
Strictly return a JSON object.""",

    # 6. Phong cách Chuyên gia Bảo trì (Maintainability Specialist)
    """Evaluate the maintainability of this method. We are looking for Shotgun Surgery and Divergent Change.
Method Analysis: Use 'method_body' and 'external_calls'.
History Analysis: Use 'pre_computed' and 'commit_messages'.
Logic: A method is smells only if ALL metrics in its category meet or exceed thresholds. Use semantic judgement to resolve borderline cases.
Response: JSON format. Include architectural_context (method_role and project_architecture).""",

    # 7. Phong cách Ngắn gọn (Minimalist / Direct)
    """Analyze the provided code and metrics for Shotgun Surgery and Divergent Change.
Input: JSON with method details, metrics, and commits.
Output: JSON with derived_metrics, semantic_analysis, and final_decision.
Rules: Check thresholds strictly. Use semantic deep-dive to verify if the co-changes are truly related to the method. Return ONLY JSON.""",

    # 8. Phong cách Tư duy logic (Logic-Heavy)
    """Determine the presence of architectural debt.
Equation for Shotgun Surgery: (fan_out >= threshold) & (co_changes >= threshold) & (history >= threshold).
Equation for Divergent Change: (history >= threshold) & (concerns >= 3) & (complexity_delta > 0).

Validate the 'file_path' to understand the method's role (e.g., Controller vs Utility). Provide a reasoning chain that explains the mismatch between raw data and semantic purpose if it exists.
Output: Valid JSON schema only.""",

    # 9. Phong cách Giám sát Git (Git Evolution Analyst)
    """Analyze how this method has evolved over time to find smells.
1. Does it change with too many other files (Shotgun Surgery)?
2. Does it change for too many different reasons (Divergent Change)?
Examine 'commit_messages' for keywords and 'total_unique_co_changed_classes' for coupling.
Provide derived metrics including change_frequency_monthly and avg_files_per_commit.
Return ONLY JSON.""",

    # 10. Phong cách Tổng quát (General Instruction)
    """You are a specialized engine for code smell analysis.
Identify if the given method exhibits SHOTGUN SURGERY or DIVERGENT CHANGE based on the provided static analysis and git pre-computed data.
Follow the provided schema for derived_metrics and semantic_analysis.
Ensure the reasoning_chain explicitly addresses the thresholds and architectural context.
Final output must be a valid JSON object without any additional text."""
]

In [ ]:
import json
import random
import copy
import re

all_data = []
pos_count = 0
neg_count = 0

file_path = "/kaggle/working/final_dataset.jsonl"

def augment_code_variable_renaming(code_snippet):
    """
    Thay đổi tên các biến trong đoạn code một cách ngẫu nhiên.
    Sử dụng Regex để nhận diện các biến phổ biến trong Java/C++.
    """
    # 1. Danh sách các từ khóa cần tránh đổi tên (Keywords của Java/C++)
    keywords = {
        'public', 'private', 'protected', 'static', 'final', 'int', 'double', 'float',
        'String', 'boolean', 'if', 'else', 'for', 'while', 'return', 'class', 'void',
        'new', 'import', 'package', 'Optional', 'override', 'this', 'true', 'false'
    }

    # 2. Tìm các định danh (identifiers) trong code
    # Regex này tìm các từ bắt đầu bằng chữ cái, dài ít nhất 3 ký tự (để tránh các từ quá ngắn)
    identifiers = set(re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', code_snippet))

    # Lọc bỏ từ khóa và các tên hàm phổ biến (thường bắt đầu bằng chữ in hoa hoặc quá ngắn)
    potential_vars = [word for word in identifiers if word not in keywords and len(word) > 2]

    # 3. Tạo ánh xạ tên biến mới (ví dụ: var_1, var_2...)
    mapping = {}
    random.shuffle(potential_vars)
    for i, var in enumerate(potential_vars):
        # Chỉ đổi tên khoảng 50-70% số biến để giữ lại một chút ngữ cảnh gốc
        if random.random() > 0.4:
            mapping[var] = f"var_{i}_{random.randint(100, 999)}"

    # 4. Thay thế trong đoạn code
    new_code = code_snippet
    for old_name, new_name in mapping.items():
        # Sử dụng \b để đảm bảo chỉ thay thế đúng từ, không thay thế một phần của từ khác
        new_code = re.sub(r'\b' + old_name + r'\b', new_name, new_code)

    return new_code

def shuffle_json_keys(obj):
    """
    Hàm đệ quy để xáo trộn ngẫu nhiên thứ tự các key trong một đối tượng JSON (Dict).
    Xử lý cả các object bị lồng sâu bên trong hoặc nằm trong mảng (List).
    """
    if isinstance(obj, dict):
        # Lấy danh sách các key và xáo trộn chúng
        keys = list(obj.keys())
        random.shuffle(keys)
        # Tạo lại dictionary với thứ tự key mới, tiếp tục đệ quy cho các giá trị bên trong
        return {k: shuffle_json_keys(obj[k]) for k in keys}
    elif isinstance(obj, list):
        # Nếu là danh sách, áp dụng đệ quy cho từng phần tử trong danh sách
        return [shuffle_json_keys(item) for item in obj]
    else:
        # Nếu là giá trị cơ bản (int, str, bool, null), giữ nguyên
        return obj

def augment_output_json(json_string):
    """
    Nhận vào chuỗi JSON đầu ra, xáo trộn thứ tự và trả về chuỗi JSON mới hợp lệ.
    """
    try:
        # 1. Chuyển chuỗi thành Dictionary
        data = json.loads(json_string)

        # 2. Xáo trộn toàn bộ key
        shuffled_data = shuffle_json_keys(data)

        # 3. Chuyển ngược lại thành chuỗi JSON (Giữ định dạng gọn nhẹ)
        return json.dumps(shuffled_data, ensure_ascii=False, separators=(',', ':'))
    except Exception as e:
        print(f"Lỗi khi xáo trộn JSON: {e}")
        return json_string # Nếu lỗi, trả về nguyên bản

def get_random_instruction():
    return random.choice(ALT_INSTRUCTIONS)

def augment_item(item):
    aug_item = copy.deepcopy(item)

    aug_item["output"] = augment_output_json(aug_item["output"])

    aug_item["instruction"] = get_random_instruction()

    try:
        input_data = json.loads(aug_item["input"])
        if "method_body" in input_data:
            input_data["method_body"] = augment_code_variable_renaming(input_data["method_body"])

        aug_item["input"] = json.dumps(input_data, ensure_ascii=False)
    except:
        pass # Giữ nguyên nếu không parse được JSON

    return aug_item

with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        item = json.loads(line)

        # 1. Kiểm tra nhãn trong phần output (đã được stringify JSON)
        try:
            # Parse nội dung của cột 'output' vì nó là một chuỗi JSON
            out_js = json.loads(item.get('output', '{}'))
            is_none_class = out_js.get('final_decision', {}).get('label') == 'none'
        except Exception:
            # Nếu parse lỗi, mặc định coi là mẫu quan trọng (Positive)
            is_none_class = False

        # 2. Phân loại và thống kê mẫu GỐC
        if is_none_class:
            neg_count += 1
            random_instruction_augmented_item = copy.deepcopy(item)
            random_instruction_augmented_item["instruction"] = get_random_instruction()
            all_data.append(random_instruction_augmented_item) # Mẫu Negative chỉ thêm 1 lần
        else:
            pos_count += 1
            # Thêm mẫu Positive 20 lần (gốc + 19 lần copy) để cân bằng
            for _ in range(20):
                aug_item = augment_item(item)

                all_data.append(aug_item)


In [ ]:
# 3. Chuyển list thành Hugging Face Dataset
dataset = Dataset.from_list(all_data)

# 4. In thống kê
print("-" * 30)
print(f"THỐNG KÊ MẪU GỐC (Trước khi nhân bản):")
print(f" - Số mẫu Positive (có Smell): {pos_count}")
print(f" - Số mẫu Negative (Nhãn 'none'): {neg_count}")
print(f" => Tỉ lệ gốc: 1 Positive / {neg_count/pos_count:.1f} Negative" if pos_count > 0 else "N/A")

print("-" * 30)
print(f"THỐNG KÊ DATASET SAU KHI XỬ LÝ:")
print(f" - Tổng số lượng mẫu trong dataset: {len(dataset)}")

print(f"THỐNG KÊ MẪU GỐC (Sau khi nhân bản):")
print(f" - Số mẫu Positive (có Smell): {pos_count*20}")
print(f" - Số mẫu Negative (Nhãn 'none'): {neg_count}")
print(f" => Tỉ lệ gốc: 1 Positive / {neg_count/(pos_count*20):.1f} Negative" if pos_count > 0 else "N/A")


def formatting_prompts_func(examples):
    # The 'system' key is not present in the dataset, so we adjust the function
    instructions = examples["instruction"]
    inputs = examples["input"] # Access the 'input' field
    outputs = examples["output"]
    texts = []

    for instr, inp, out in zip(instructions, inputs, outputs):
        # Combine 'instruction' and 'input' into the 'user' role
        # as 'instruction' contains the system persona and general guidelines,
        # and 'input' contains the specific query.
        messages = [
            {"role": "user", "content": f"{instr}\n\n{inp}"},
            {"role": "assistant", "content": out}
        ]
        # apply_chat_template sẽ tự thêm thẻ <｜User｜>, [INST]... tùy vào model bạn load
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        texts.append(text)

    return { "text" : texts, }

# Áp dụng vào dataset
dataset = dataset.map(formatting_prompts_func,batched=True)

In [ ]:
print(f"Số lượng mẫu trong dataset: {len(dataset)}")
print(f"Các cột hiện có: {dataset.column_names}")

if len(dataset) > 0:
    # Kiểm tra thử nội dung mẫu đầu tiên
    first_sample = dataset[0]
    print("Nội dung mẫu đầu tiên:")
    # Thay 'text' bằng tên cột bạn dùng trong dataset_text_field
    print(first_sample.get("text", "CỘT 'text' KHÔNG TỒN TẠI!"))
else:
    print("CẢNH BÁO: Dataset của bạn đang rỗng!")

In [ ]:
from trl import SFTConfig, SFTTrainer
from transformers import TrainingArguments
from transformers import EarlyStoppingCallback


callbacks = [EarlyStoppingCallback(early_stopping_patience=3)]

sft_config = SFTConfig(
    max_seq_length = max_seq_length,
    dataset_text_field = "text",
    completion_only_loss = True,
    packing = True,
    group_by_length = True,
    neftune_noise_alpha = 15,     # NEFTune mức vừa
    dataset_num_proc = 2,
)

training_args = TrainingArguments(
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 16,
    warmup_steps = 5,
    warmup_ratio = 0.1, # Tăng warmup để tránh sốc gradient
    max_steps = 30,
    # num_train_epochs = 1,
    learning_rate = 2e-4,
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 1,
    neftune_noise_alpha=15,
    optim = "adamw_8bit",
    weight_decay = 0.1,
    lr_scheduler_type = "cosine",
    seed = 3407,
    output_dir = "outputs",
    metric_for_best_model = "eval_loss",
    label_smoothing_factor = 0.1,
)

dataset_split = dataset.train_test_split(test_size=0.1)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_split["train"],
    eval_dataset = dataset_split["test"], # Thêm tập kiểm thử
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = training_args,
)

trainer_stats = trainer.train()

In [ ]:
FastLanguageModel.for_inference(model)

prompt = dataset[0]["text"].split("<|im_start|>assistant")[0] + "<|im_start|>assistant"

inputs = tokenizer([prompt], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 8192)

In [ ]:
# 1. Lưu dưới dạng LoRA Adapters (Nhẹ, dùng để load lại nhanh)
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

# 2. Xuất sang GGUF (Tối ưu nhất cho CPU và RAM thấp)
# Định dạng 'q4_k_m' là sự cân bằng tốt nhất giữa tốc độ và độ chính xác
model.save_pretrained_gguf("model_gguf_q4", tokenizer, quantization_method = "q4_k_m")

# 3. Xuất sang Merged 16bit (Tối ưu cho GPU Inference như vLLM)
model.save_pretrained_merged("model_merged_16bit", tokenizer, save_method = "merged_16bit")